In [8]:
import sys
sys.path.insert(1, '../../../scripts/')

import cobra
import gc
import pickle
from tqdm import tqdm
from core.model import ME_Model
from core.reaction import ME_Reaction
from macromolecules.RNA import mRNA
import pandas as pd
import numpy as np
import sympy
import multiprocessing
from macromolecules.RNA import pre_mRNA
import itertools


from utils import parameters as params
import copy


from expression import build_me_model

lp_path = '/data2/hratch/human_me/test_lp/'

No objective coefficients in model. Unclear what should be optimized


# Add sinks

In [9]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_sink(metabolite_ids, mu_val = 1e-9, model = None):
    '''metabolite ids is a list of metabolite ids to add as sinks and test model feasibility'''
    
    try:
        if model is None:
#             with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
#                 model = pickle.load(handle) # boundary infeasible
            with open('/data2/hratch/human_me/test_lp/' + 'dilution_coupled.pickle', 'rb') as handle:
                model = pickle.load(handle) # real infeasible
            
        model.initialize_solver(solver_type='qminos', precision='quad')

        metabolite_ids = sorted(set(metabolite_ids))
        reactions = [r.copy() for r in tqdm(model.reactions)]
        for m_id in metabolite_ids:
            r = cobra.Reaction('SK_' + m_id)
            r.bounds = (-1000,1000)
            r.add_metabolites({model.metabolites.get_by_id(m_id).copy(): -1})
            reactions.append(r)

        print('Generate model')
        tm = ME_Model(id_or_model = 'tm', m_model = params.human_model)
        tm.add_reactions(reactions)
        tm.initialize_solver(solver_type='qminos', precision='quad')

        print('Solve')
        sln, stat, _ = tm.solve_lp(mu_val = mu_val)

        store = {'model': tm, 'sln': sln, 'stat': stat, 'sinks': metabolite_ids, 
                'infeasible_reactions': tm.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)}
    except:
        store = {'model': float('nan'), 'sln': float('nan'), 'stat': float('nan'), 'sinks': metabolite_ids, 
                'infeasible_reactions': float('nan')}
    return store

def par_sink(m_id_lists, n_cores):
    '''m_id_lists is a list of lists'''
    pool = multiprocessing.Pool(processes = n_cores)
    try:
        stores = pool.map(add_sink, m_id_lists)
        pool.close()
        pool.join()
        gc.collect()
        return stores
    except:
        pool.close()
        pool.join()
        gc.collect()
        raise ValueError('par failed')
    

In [3]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_demand(metabolite_ids, mu_val = 1e-9, model = None):
    '''metabolite ids is a list of metabolite ids to add as sinks and test model feasibility'''
    
    try:
        if model is None:
            with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
                model = pickle.load(handle) # boundary infeasible
        model.initialize_solver(solver_type='qminos', precision='quad')

        metabolite_ids = sorted(set(metabolite_ids))
        reactions = [r.copy() for r in tqdm(model.reactions)]
        for m_id in metabolite_ids:
            r = cobra.Reaction('DM_' + m_id)
            r.bounds = (0,1000)
            r.add_metabolites({model.metabolites.get_by_id(m_id).copy(): -1})
            reactions.append(r)

        print('Generate model')
        tm = ME_Model(id_or_model = 'tm', m_model = params.human_model)
        tm.add_reactions(reactions)
        tm.initialize_solver(solver_type='qminos', precision='quad')

        print('Solve')
        sln, stat, _ = tm.solve_lp(mu_val = mu_val)

        store = {'model': tm, 'sln': sln, 'stat': stat, 'sinks': metabolite_ids, 
                'infeasible_reactions': tm.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)}
    except:
        store = {'model': float('nan'), 'sln': float('nan'), 'stat': float('nan'), 'sinks': metabolite_ids, 
                'infeasible_reactions': float('nan')}
    return store

def par_demand(m_id_lists, n_cores):
    '''m_id_lists is a list of lists'''
    pool = multiprocessing.Pool(processes = n_cores)
    try:
        stores = pool.map(add_demand, m_id_lists)
        pool.close()
        pool.join()
        gc.collect()
        return stores
    except:
        pool.close()
        pool.join()
        gc.collect()
        raise ValueError('par failed')
    

In [3]:
boundary_metab = [m for m in params.human_model.metabolites if m.compartment == 'b']
test_m = []
for m_b in tqdm(boundary_metab):
    consider = False
    consider_2 = False
    
    reactions_b = m_b.reactions
    for r in reactions_b:
        if 'e' in r.compartments and len(r.genes) == 0:
            consider = True
    
    if consider:
        m_e = params.human_model.metabolites.get_by_id('_'.join(m_b.id.split('_'))[:-1] + 'e')
        reactions_e = m_e.reactions
        for r in reactions_e:
            if 'c' in r.compartments and len(r.genes) == 0:
                consider_2 = True 
    
    if consider_2:
        test_m.append(m_e.id)


100%|██████████| 84/84 [00:00<00:00, 58186.88it/s]


In [13]:
test_m = sorted(set(['13_cis_retnglc_e', '3bcrn_e', '3ivcrn_e', 'HC00250_e', 'HC00342_e', 'ddca_e', 'Rtotal_e', 'acald_e', 
          'ahcys_e', 'asp_L_e', 'atp_e', 'bhb_e', 'c4crn_e', 'cit_e', 'crvnc_e', 'dcmp_e', 'glu_L_e', 'glyc_e', 
          'h2o2_e', 'h_e', 'o2_e', 'o2s_e', 'pe_hs_e', 'pglyc_hs_e', 'pro_L_e', 'ps_hs_e', 'retn_e', 
          'sbt_D_e', 'sph1p_e', 'utp_e', 'xmp_e']))
test_metab = [[m_id.replace('_e', '_c')] for m_id in test_m]


In [17]:
stores3 = par_sink(m_id_lists = test_metab, n_cores = len(test_m))

In [20]:
test_metab = ['7dhchsterol_r', 'bhb_c', 'bhb_e', 'chol_b', 'chol_e', 'chsterol_r', 'h_c', 'h_e', 'h_r', 'lac_L_b', 
'lac_L_c', 'lac_L_e', 'nadp_r', 'nadph_r', 'srtn_c', 'srtn_e']
test_metab = ['gal_e', 'gal_c', 'cl_e', 'na1_e', 'abut_c', 'na1_c']

In [21]:
store = add_sink(test_metab)

100%|██████████| 12650/12650 [01:56<00:00, 108.14it/s]


In [26]:
for store in stores3:
    if store['stat'].max()<1:
        m_id = store['sinks'][0]
        print(m_id)
        try:
            print(store['sln'][store['model'].reactions.index('SK_' + m_id)])
            print(store['model'].reactions.get_by_id('SK_' + m_id).reaction)
            print('-------')
        except: 
            print('-------')
        

Rtotal_c
2.792295707079784e-07
Rtotal_c <=> 
-------
bhb_c
-2.2877365672362936e-07
bhb_c <=> 
-------
cit_c
-2.2877384226969247e-07
cit_c <=> 
-------
glu_L_c
1.3755636112783917e-07
glu_L_c <=> 
-------
h_c
2.286153207108273e-07
h_c <=> 
-------


In [10]:
m_ids = ['Rtotal_c', 'bhb_c', 'cit_c', 'glu_L_c', 'h_c']
m_ids = [m_ids] + [[m_id] for m_id in m_ids]

In [31]:
# cit_mod = [store['model'] for store in stores3 if store['sinks'] == ['cit_c']][0]
# bhb_mod = [store['model'] for store in stores3 if store['sinks'] == ['bhb_c']][0]

# cit_sln = [store['sln'] for store in stores3 if store['sinks'] == ['cit_c']][0]
# bhb_sln = [store['sln'] for store in stores3 if store['sinks'] == ['bhb_c']][0]

# with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
#     fail = pickle.load(handle) # boundary infeasible
# fail.initialize_solver()
# slnf, statf, _ = fail.solve_lp(mu_val = 1e-9)

# res_ = {'cit': {'model': cit_mod, 'sln': cit_sln}, 
#       'bhb': {'model': bhb_mod, 'sln': bhb_sln}, 
#       'fail': {'model': fail, 'sln': slnf}}

# res_df = pd.DataFrame(columns = res_.keys(), index = [r.id for r in res_['fail']['model'].reactions])
# for key in res_df.columns:
#     for r_id in res_df.index:
#         res_df.loc[r_id, key] = res_[key]['sln'][res_[key]['model'].reactions.index(r_id)]
# summ = res_df[res_df.apply(lambda x: sum(x), axis = 1) != 0]

In [61]:
# fail.metabolites.get_by_id('glu_L_c').reactions

In [62]:
# r_ids = [r.id for r in fail.metabolites.get_by_id('Rtotal_c').reactions]

# summ.loc[[r_id for r_id in r_ids if r_id in summ.index.tolist()],:]

In [63]:
# fail.reactions.get_by_id('glu_L_c').reaction